# 02 · Limpieza, split temporal y análisis exploratorio (EDA)
Partimos del tablón analítico del notebook 01 (una fila por producto, tienda y día). Orden de trabajo: **limpiar → separar por fecha → explorar solo el entrenamiento**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

DATOS = (Path("..") / "02_datos").resolve()
RUTA = DATOS / "03_Entrenamiento"
VALIDACION = DATOS / "02_Validacion"

# El archivo trae date como índice; la pasamos a columna para trabajar con ella
tablon = pd.read_parquet(RUTA / "tablon_analitico.parquet").reset_index()
print(tablon.shape)
tablon.dtypes

## 1. Limpieza: nulos de `sell_price`
Antes de decidir qué hacer con un nulo hay que entender **por qué** está. Miramos qué filas son.

In [ ]:
sin_precio = tablon["sell_price"].isna()

tablon.loc[sin_precio].groupby(["item_id", "store_id"]).agg(
    filas=("d", "size"),
    desde=("date", "min"),
    hasta=("date", "max"),
    ventas=("ventas", "sum"),
)

Los nulos son de **un solo producto** (`FOODS_3_288`), en las primeras semanas de 2013 y con **0 ventas**. Ese producto todavía no estaba a la venta en esas tiendas.

Esas filas no son "días en que no se vendió nada", son "días en que el producto no existía". Si las dejamos, el modelo aprende ceros falsos. Son solo 71 filas, así que las **eliminamos**. Un `assert` confirma que no perdemos ninguna venta real.

Esta limpieza es una regla fija (no calcula nada a partir de los datos), por eso se puede hacer antes del split sin filtrar información del futuro.

In [ ]:
assert (tablon.loc[sin_precio, "ventas"] == 0).all()

tablon_limpio = tablon.loc[~sin_precio].reset_index(drop=True)

# Los nulos de eventos significan "ese día no hubo evento"
cols_evento = ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]
tablon_limpio[cols_evento] = tablon_limpio[cols_evento].fillna("Ninguno")
tablon_limpio["hay_evento"] = tablon_limpio["event_name_1"] != "Ninguno"

print(f"{len(tablon)} -> {len(tablon_limpio)} filas | nulos restantes: {int(tablon_limpio.isna().sum().sum())}")
tablon_limpio.set_index("date").to_parquet(RUTA / "tablon_limpio.parquet")

## 2. Split temporal: entrenamiento, validación y reserva
Lo separamos **ahora**, antes de explorar y de crear variables. Si miramos todos los datos para decidir qué construir, la validación deja de ser una prueba honesta.

**Se separa por fecha, nunca al azar.** Un split aleatorio mezclaría días futuros dentro del entrenamiento: el modelo "vería" el mañana para predecir el ayer y la validación saldría mucho mejor de lo que será en la realidad.

Cortes elegidos:

| Parte | Período | Para qué |
|---|---|---|
| `train` | hasta el 2015-11-10 | entrenar y explorar |
| `valid` | 2015-11-11 a 2015-12-08 (28 días) | comparar modelos. Son 4 semanas completas, así que cada día de la semana aparece 4 veces |
| `reserva` | 2015-12-09 a 2015-12-31 (23 días) | no se toca hasta el final. Coincide con `DatosParaProduccion.csv` de la carpeta `Producción/`, que predice 8 días hacia adelante |

In [ ]:
FIN_TRAIN = "2015-11-10"
FIN_VALID = "2015-12-08"

es_train = tablon_limpio["date"] <= FIN_TRAIN
es_valid = (tablon_limpio["date"] > FIN_TRAIN) & (tablon_limpio["date"] <= FIN_VALID)
es_reserva = tablon_limpio["date"] > FIN_VALID

train = tablon_limpio[es_train]
valid = tablon_limpio[es_valid]
reserva = tablon_limpio[es_reserva]

# Controles: nada se pierde, nada se repite y todo el entrenamiento es anterior a la validación
assert len(train) + len(valid) + len(reserva) == len(tablon_limpio)
assert train["date"].max() < valid["date"].min()
assert valid["date"].max() < reserva["date"].min()

pd.DataFrame(
    {nombre: {"filas": len(df), "dias": df["date"].nunique(), "desde": df["date"].min().date(), "hasta": df["date"].max().date()}
     for nombre, df in [("train", train), ("valid", valid), ("reserva", reserva)]}
).T

In [ ]:
train.set_index("date").to_parquet(RUTA / "train.parquet")
valid.set_index("date").to_parquet(VALIDACION / "validacion.parquet")
reserva.set_index("date").to_parquet(VALIDACION / "reserva.parquet")
print("✅ Guardados: train.parquet, validacion.parquet, reserva.parquet")

## 3. EDA: ¿cómo se comportan las ventas?
A partir de acá se usa **solo `train`**.

### Evolución mensual
Suma de unidades vendidas de las 20 series, mes a mes. Buscamos **tendencia** y **estacionalidad**.

In [ ]:
mensual = train.groupby(train["date"].dt.to_period("M"))["ventas"].sum()
mensual.index = mensual.index.to_timestamp()
mensual = mensual.iloc[:-1]  # noviembre de 2015 está incompleto (el entrenamiento corta el día 10)

fig, ax = plt.subplots(figsize=(12, 4))
mensual.plot(ax=ax, marker="o")
ax.set_title("Unidades vendidas por mes")
ax.set_ylabel("Unidades")
plt.show()

**Observación:** los meses más bajos son febrero, noviembre y diciembre (12 a 15 mil unidades) y **julio** se mantiene alto los tres años (20 a 21 mil). Septiembre de 2013 (26,4 mil) es un pico que se destaca y habría que revisarlo. El total de 2014 (208 mil) es un 5 % menor que el de 2013 (218 mil); 2015 no se compara porque el entrenamiento no llega a diciembre.

### Por tienda y por producto
¿Quién vende más? Si una tienda o un producto domina, el modelo tiene que tratarlo distinto.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

train.groupby("store_id")["ventas"].sum().plot.bar(ax=axes[0], rot=0)
axes[0].set_title("Ventas totales por tienda")

train.groupby("item_id")["ventas"].sum().sort_values().plot.barh(ax=axes[1])
axes[1].set_title("Ventas totales por producto")

plt.tight_layout()
plt.show()

**Observación:** `CA_3` concentra el **80 %** de las ventas (482 mil unidades contra 121 mil de `CA_4`). Entre los productos, `FOODS_3_090` pesa el 27 % y los tres primeros (`090`, `586`, `120`) suman el 54 %. El volumen es muy desparejo entre series.

### Día de la semana
Promedio de unidades por fila (producto y tienda) según el día.

In [ ]:
orden = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=train, x="weekday", y="ventas", order=orden, errorbar=None, ax=ax)
ax.set_title("Venta promedio por día de la semana")
ax.set_xlabel("")
plt.show()

**Observación:** sábado y domingo venden un **41 % más** que el miércoles (35,1 y 34,6 contra 24,9 unidades). Hay una estacionalidad semanal clara, y `weekday` va a ser una variable importante para el modelo.

### Efecto de los eventos
¿Se vende distinto en días con evento (feriados, fiestas) que en días normales?

In [ ]:
print(train.groupby("hay_evento")["ventas"].mean().round(1))
print()
print(train.groupby("event_type_1")["ventas"].mean().round(1).sort_values(ascending=False))

**Observación:** en promedio, un día con evento vende lo mismo que uno sin evento (28,9 vs 29,0 unidades). Los eventos no mueven las ventas a este nivel agregado. Con solo 20 series, un efecto puntual en un producto puede quedar tapado por el promedio.

## Resumen
- Se eliminaron 71 filas de un producto que todavía no estaba a la venta. No se perdió ninguna venta real.
- El split es **temporal**: `train` hasta el 2015-11-10, `valid` de 28 días y `reserva` de 23 días sin tocar.
- Hay estacionalidad **semanal** (fin de semana arriba) y **anual** (julio alto; febrero, noviembre y diciembre bajos).
- Las ventas están muy concentradas en una tienda y en pocos productos.
- Los eventos no muestran un efecto visible en el promedio.

Siguiente paso: construir variables (rezagos, medias móviles) sobre `train` y entrenar un primer modelo.